# Лабораторная работа: NoSQL базы данных. Key-Value хранилища. Redis

**Цель:** получение практических навыков работы с key-value хранилищем Redis.  
**Инструменты:** Docker (Redis 7), Python 3, библиотека `redis-py`

> Redis запускается через Docker: `docker compose up -d` в папке `lab_1_redis/`

In [ ]:
import redis

r = redis.Redis(host='localhost', port=6379, decode_responses=True)
print('Ping:', r.ping())
print('Redis version:', r.info('server')['redis_version'])

---
## 2.1.1 Строки и срок жизни (TTL)

Цель: научиться записывать, читать и удалять данные, устанавливать время жизни ключа.

In [ ]:
# SET — создание записи
r.set('greeting', 'Hello, Redis!')
print('SET greeting -> OK')

# GET — чтение значения
value = r.get('greeting')
print(f'GET greeting -> {value}')

In [ ]:
# TTL до установки срока жизни (вернёт -1 — ключ существует, но без TTL)
print(f'TTL greeting (before EXPIRE) -> {r.ttl("greeting")}')

# EXPIRE — установить время жизни 30 секунд
r.expire('greeting', 30)

# TTL после установки
print(f'TTL greeting (after EXPIRE 30s) -> {r.ttl("greeting")} сек')

In [ ]:
# DEL — удаление ключа
r.delete('greeting')
print(f'GET после DEL -> {r.get("greeting")}')

# TTL для несуществующего ключа (вернёт -2)
print(f'TTL несуществующего ключа -> {r.ttl("greeting")}')
# -1 = ключ есть, но без TTL
# -2 = ключ не существует

---
## 2.1.2 Хэши — профили пользователей

Цель: изучить работу с Redis Hash (объекты).  
**Хэши предпочтительнее строк** когда нужно хранить объект с несколькими полями и обновлять их по отдельности.

In [ ]:
# HSET — создание профиля пользователя
r.hset('user:1', mapping={'name': 'Иван', 'age': 25, 'city': 'Москва'})
print('HSET user:1 -> профиль создан')

# HGET — получить отдельное поле
name = r.hget('user:1', 'name')
print(f'HGET user:1 name -> {name}')

# HGETALL — получить весь профиль
profile = r.hgetall('user:1')
print(f'HGETALL user:1 -> {profile}')

In [ ]:
# HINCRBY — увеличить возраст на 1
new_age = r.hincrby('user:1', 'age', 1)
print(f'HINCRBY age +1 -> age = {new_age}')

# HDEL — удалить поле city
r.hdel('user:1', 'city')
profile = r.hgetall('user:1')
print(f'HGETALL после HDEL city -> {profile}')
# city больше не присутствует в профиле

---
## 2.1.3 Списки — блог / лента активности

Цель: использовать Redis List для хранения упорядоченных данных.

- **LPUSH + RPOP** = очередь FIFO (добавляем слева, читаем справа)
- **RPUSH + LPOP** = та же очередь FIFO с другого конца  
- **LTRIM** удобен для ограничения длины ленты (например, «хранить последние N записей»)

In [ ]:
# Очистим ключ на случай повторного запуска
r.delete('posts:recent')

# LPUSH — добавить три записи в начало списка
r.lpush('posts:recent', 'Пост 1')
r.lpush('posts:recent', 'Пост 2')
r.lpush('posts:recent', 'Пост 3')

# LRANGE — вывести все элементы (0 = первый, -1 = последний)
posts = r.lrange('posts:recent', 0, -1)
print(f'LRANGE posts:recent 0 -1 -> {posts}')
# Порядок обратный: Пост 3 добавлен последним -> стоит первым (head)

In [ ]:
# LINDEX — получить элемент по индексу без удаления
latest = r.lindex('posts:recent', 0)
print(f'LINDEX 0 (последний добавленный) -> {latest}')

# RPOP — удалить последний элемент (хвост списка)
removed = r.rpop('posts:recent')
print(f'RPOP (удалён из хвоста) -> {removed}')
print(f'Список после RPOP -> {r.lrange("posts:recent", 0, -1)}')

In [ ]:
# Добавим элементы обратно для демонстрации LTRIM
r.lpush('posts:recent', 'Пост 1')
print(f'Перед LTRIM -> {r.lrange("posts:recent", 0, -1)}')

# LTRIM — обрезать список до двух элементов
r.ltrim('posts:recent', 0, 1)
print(f'После LTRIM 0 1 -> {r.lrange("posts:recent", 0, -1)}')

---
## 2.1.4 Множества — уникальные теги

Цель: работа с коллекциями уникальных элементов.

- **Set vs List**: Set хранит только уникальные элементы без порядка; List допускает дубликаты и сохраняет порядок вставки.
- Пересечение множеств: `SINTER post:101:tags post:102:tags` — общие теги двух постов.

In [ ]:
r.delete('post:101:tags')

# SADD — добавить теги (дубликат 'redis' не добавится повторно)
r.sadd('post:101:tags', 'redis', 'database', 'nosql')
r.sadd('post:101:tags', 'redis', 'tutorial')  # 'redis' уже есть — игнорируется

# SMEMBERS — получить все теги
tags = r.smembers('post:101:tags')
print(f'SMEMBERS -> {tags}')

In [ ]:
# SISMEMBER — проверить наличие тега
print(f'SISMEMBER "mongodb" -> {r.sismember("post:101:tags", "mongodb")}')
print(f'SISMEMBER "redis"   -> {r.sismember("post:101:tags", "redis")}')

# SREM — удалить тег
r.srem('post:101:tags', 'nosql')
print(f'После SREM "nosql" -> {r.smembers("post:101:tags")}')

# SCARD — количество тегов
print(f'SCARD -> {r.scard("post:101:tags")}')

In [ ]:
# Демонстрация SINTER — пересечение тегов двух постов
r.sadd('post:102:tags', 'redis', 'python', 'tutorial')
common = r.sinter('post:101:tags', 'post:102:tags')
print(f'Общие теги постов 101 и 102: {common}')

---
## 2.2 Мини-проект «Чат» на Python (Pub/Sub)

Реализация чата с использованием Redis Pub/Sub:
- **Топик:** `RUDN_CHAT`
- **История:** список `RUDN_CHAT_HISTORY` (последние 20 сообщений)
- **Формат сообщения:** JSON `{username, message, timestamp}`
- Чтение и отправка выполняются в разных потоках (`threading`)

In [ ]:
import json
import threading
import time
from datetime import datetime
import redis

CHANNEL = 'RUDN_CHAT'
HISTORY_KEY = 'RUDN_CHAT_HISTORY'
HISTORY_MAX = 20


def save_message(rc: redis.Redis, username: str, message: str) -> None:
    """Сохранить сообщение в историю (список, макс. 20 элементов)."""
    payload = json.dumps({
        'username': username,
        'message': message,
        'timestamp': datetime.now().strftime('%H:%M:%S')
    }, ensure_ascii=False)
    rc.lpush(HISTORY_KEY, payload)
    rc.ltrim(HISTORY_KEY, 0, HISTORY_MAX - 1)  # Храним только последние 20


def load_history(rc: redis.Redis) -> list:
    """Загрузить историю (последние 20 сообщений, хронологически)."""
    raw = rc.lrange(HISTORY_KEY, 0, HISTORY_MAX - 1)
    return [json.loads(m) for m in reversed(raw)]


def send_message(rc: redis.Redis, username: str, message: str) -> None:
    """Сохранить в историю и опубликовать в канал."""
    save_message(rc, username, message)
    payload = json.dumps({
        'username': username,
        'message': message,
        'timestamp': datetime.now().strftime('%H:%M:%S')
    }, ensure_ascii=False)
    rc.publish(CHANNEL, payload)


def receive_messages(pubsub, stop_event: threading.Event) -> None:
    """Поток: слушаем входящие сообщения из канала."""
    for raw_msg in pubsub.listen():
        if stop_event.is_set():
            break
        if raw_msg['type'] == 'message':
            msg = json.loads(raw_msg['data'])
            print(f"[{msg['timestamp']}] {msg['username']}: {msg['message']}")


def run(username: str, messages: list, delay: float = 0.5) -> None:
    """
    Запуск чата.
    
    - Показывает историю
    - Запускает поток-подписчик
    - Отправляет список сообщений с паузой
    - Останавливает подписчик
    """
    rc = redis.Redis(host='localhost', port=6379, decode_responses=True)
    pubsub = rc.pubsub()
    pubsub.subscribe(CHANNEL)

    # --- история ---
    history = load_history(rc)
    if history:
        print('=== История чата ===')
        for m in history:
            print(f"  [{m['timestamp']}] {m['username']}: {m['message']}")
        print('===================')
    else:
        print('История пуста.')

    # --- поток приёма ---
    stop_event = threading.Event()
    listener = threading.Thread(
        target=receive_messages,
        args=(pubsub, stop_event),
        daemon=True
    )
    listener.start()

    # --- отправка сообщений ---
    for msg in messages:
        time.sleep(delay)
        print(f'>>> Отправка: {msg}')
        send_message(rc, username, msg)

    time.sleep(0.5)  # дать подписчику время принять последнее сообщение
    stop_event.set()
    pubsub.unsubscribe(CHANNEL)
    print('\n[Сеанс завершён]')


print('Чат определён. Запустите следующую ячейку для демонстрации.')

In [ ]:
# Очистить историю перед демонстрацией (опционально)
r.delete(HISTORY_KEY)

# Симуляция: пользователь «Алиса» отправляет несколько сообщений
run(
    username='Алиса',
    messages=[
        'Привет всем!',
        'Redis Pub/Sub работает отлично',
        'До свидания!'
    ]
)

In [ ]:
# Второй участник «Боб» видит историю и добавляет своё сообщение
run(
    username='Боб',
    messages=['Привет, Алиса! Я тоже здесь']
)

In [ ]:
# Просмотр итоговой истории чата
print('=== Финальная история ===' )
for m in load_history(r):
    print(f"  [{m['timestamp']}] {m['username']}: {m['message']}")